In [2]:
import os
import zipfile
import rasterio
from rasterio.merge import merge
from rasterio.crs import CRS
import numpy as np
import glob
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

In [4]:
for year in [2020]:

    region = 'Thames'


    extract_folder = f"G:\DTM_ref\{region}/{region}_{year}_test/Zemp_extracted4"
    reproject_folder = f"G:\DTM_ref\{region}/{region}_{year}_test/Zemp_proj4"
    os.makedirs(extract_folder, exist_ok=True)
    os.makedirs(reproject_folder, exist_ok=True)

    zip_folder = f'G:\DTM_ref\{region}/{region}_{year}_test'
    output_tiff = f'G:\DTM_ref\{region}/{region}_{year}_test/{region}_merge_{year}.tif'
    output_masked_tiff =  f'G:\DTM_ref\{region}/{region}_{year}_test/{region}_masked_{year}.tif'
    nodata_value =  -999



    # #1️⃣ 解压所有 ZIP 文件
    for zip_file in glob.glob(os.path.join(zip_folder, "*.zip")):
        with zipfile.ZipFile(zip_file, "r") as zip_ref:
            zip_ref.extractall(extract_folder)

    # 2️⃣ 读取解压后的 TIFF 影像，并转换坐标系（若需要）
    tiff_files = glob.glob(os.path.join(extract_folder, "*.tif"))
    reprojected_files = []

    for tiff in tiff_files:
        with rasterio.open(tiff) as src:
            src_crs = src.crs
            dst_crs = CRS.from_epsg(32630)  # WGS 1984 UTM Zone 30N

            if src_crs != dst_crs:
                # 计算重投影参数
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=(10, 10)
                )

                # 更新元数据
                out_meta = src.meta.copy()
                out_meta.update({
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height,
                    "nodata": nodata_value,  # 显式设置 nodata
                    "dtype": "float32"  # 确保数据类型
                })

                # 生成新的文件路径
                reprojected_tiff = os.path.join(reproject_folder, os.path.basename(tiff))
                reprojected_files.append(reprojected_tiff)

                # 执行重投影
                with rasterio.open(reprojected_tiff, "w", **out_meta) as dst:
                    for i in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, i),
                            destination=rasterio.band(dst, i),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            src_nodata=src.nodata,  # 读取源影像的 nodata
                            dst_nodata=nodata_value  # 设定目标影像的 nodata
                        )
            else:
                reprojected_files.append(tiff)  # 若已是 WGS 1984，直接使用

    # 3️⃣ 影像拼接
    datasets = [rasterio.open(f) for f in reprojected_files]
    mosaic, out_trans = merge(datasets, nodata=nodata_value)  # 避免 nodata 警告

    # 4️⃣ 保存拼接影像
    out_meta = datasets[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": nodata_value  # 确保 nodata 统一
    })

    with rasterio.open(output_tiff, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"✅ 拼接完成，影像已保存至: {output_tiff}")

    # 设置路径
    input_path = output_tiff
    output_path = output_masked_tiff  # 保存的新路径

    # 打开并读取数据
    with rasterio.open(input_path) as src:
        data = src.read(1)  # 读取第一波段
        meta = src.meta.copy()

    # 应用条件：把小于 -5 或大于 5 的值改成 6
    data[(data < -5) | (data > 5)] = 6

    # 保存修改后的数据
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(data, 1)

    print(f"✅ 修改后的影像已保存到: {output_path}")

✅ 拼接完成，影像已保存至: G:\DTM_ref\Thames/Thames_2020_test/Thames_merge_2020.tif
✅ 修改后的影像已保存到: G:\DTM_ref\Thames/Thames_2020_test/Thames_masked_2020.tif
